In [13]:
# ============================================
# Customer Churn Prediction using ANN (PyTorch)
# ============================================

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

# Fix randomness for consistent results
torch.manual_seed(42)

print("Libraries imported successfully")

Libraries imported successfully


In [14]:
# Load dataset

df = pd.read_csv("Churn_Modelling.csv")

print("Dataset loaded successfully")

df.head()

Dataset loaded successfully


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [15]:
print("Dataset shape:", df.shape)

df.info()

Dataset shape: (10000, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [16]:
# Remove unnecessary columns

df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Encode Gender

label_encoder = LabelEncoder()

df["Gender"] = label_encoder.fit_transform(df["Gender"])

# One-hot encode Geography

df = pd.get_dummies(df, columns=["Geography"], drop_first=True)

print("Preprocessing completed")

df.head()

Preprocessing completed


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,False,False
1,608,0,41,1,83807.86,1,0,1,112542.58,0,False,True
2,502,0,42,8,159660.80,3,1,0,113931.57,1,False,False
3,699,0,39,1,0.00,2,0,0,93826.63,0,False,False
4,850,0,43,2,125510.82,1,1,1,79084.10,0,False,True


In [17]:
# Features and target

X = df.drop("Exited", axis=1).values

y = df["Exited"].values

print("Features shape:", X.shape)

Features shape: (10000, 11)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(

    X, y, test_size=0.2, random_state=42

)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (8000, 11)
Test size: (2000, 11)


In [19]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

print("Scaling completed")

Scaling completed


In [20]:
X_train = torch.FloatTensor(X_train)

X_test = torch.FloatTensor(X_test)

y_train = torch.FloatTensor(y_train).view(-1,1)

y_test = torch.FloatTensor(y_test).view(-1,1)

print("Converted to tensors")

Converted to tensors


In [21]:
class ANN_Model(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.fc1 = nn.Linear(input_size, 32)

        self.fc2 = nn.Linear(32, 16)

        self.fc3 = nn.Linear(16, 8)

        self.fc4 = nn.Linear(8, 1)

        self.relu = nn.ReLU()

        self.sigmoid = nn.Sigmoid()


    def forward(self, x):

        x = self.relu(self.fc1(x))

        x = self.relu(self.fc2(x))

        x = self.relu(self.fc3(x))

        x = self.sigmoid(self.fc4(x))

        return x


input_size = X_train.shape[1]

model = ANN_Model(input_size)

print(model)

ANN_Model(
  (fc1): Linear(in_features=11, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=8, bias=True)
  (fc4): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [22]:
criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss and optimizer ready")

Loss and optimizer ready


In [23]:
epochs = 300

for epoch in range(epochs):

    outputs = model(X_train)

    loss = criterion(outputs, y_train)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch+1) % 50 == 0:

        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 50, Loss: 0.5824
Epoch 100, Loss: 0.4653
Epoch 150, Loss: 0.4257
Epoch 200, Loss: 0.3974
Epoch 250, Loss: 0.3540
Epoch 300, Loss: 0.3314


In [24]:
with torch.no_grad():

    predictions = model(X_test)

    predictions = (predictions > 0.5).float()

    accuracy = accuracy_score(y_test, predictions)

print("Final Accuracy:", accuracy*100, "%")

Final Accuracy: 86.05000000000001 %
